In [7]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install lightning

  Using cached lightning-2.6.1-py3-none-any.whl.metadata (44 kB)
  Using cached lightning_utilities-0.15.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached pytorch_lightning-2.6.1-py3-none-any.whl.metadata (21 kB)
Using cached lightning-2.6.1-py3-none-any.whl (853 kB)
Using cached lightning_utilities-0.15.3-py3-none-any.whl (31 kB)
Using cached torchmetrics-1.9.0-py3-none-any.whl (983 kB)
Using cached pytorch_lightning-2.6.1-py3-none-any.whl (857 kB)
Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch
import sys
from pathlib import Path

sys.path.append(str(Path("../scripts").resolve()))

from dataset import EarthCARELightningDataset

from datamodule import EarthCARELightningDataModule
from models.unet import UNet

In [10]:
INPUT_VARS = [
    "ice_water_content",
    "ice_mass_flux",
    "ice_effective_radius",
    "ice_median_volume_diameter",
    "ice_riming_factor",
    "rain_rate",
    "rain_water_content",
    "rain_median_volume_diameter",
    "liquid_water_content",
    "liquid_number_concentration",
    "liquid_effective_radius",
    "aerosol_number_concentration",
    "aerosol_mass_content",
    "doppler_velocity_best_estimate",
    "sedimentation_velocity_best_estimate",
    "spectrum_width_integrated",
    "reflectivity_no_attenuation_correction",
    "reflectivity_corrected",
    "multiple_scattering_status",
    "simplified_convective_classification",
]

TARGET_VARS = [
    "lightning_count_2p5",
]


def main():
    dataset_path = "../scripts/dataset"

    dm = EarthCARELightningDataModule(
        data_dir=dataset_path,
        input_vars=INPUT_VARS,
        target_vars=TARGET_VARS,
        batch_size=4,
        num_workers=0,
        pin_memory=False,
        fill_value=0.0,
        norm_with_train=True,
        # seed=42,
    )

    
    dm.setup()

    print("\nSplit sizes:")
    print("train:", len(dm.train_dataset))
    print("val  :", len(dm.val_dataset))
    print("test :", len(dm.test_dataset))

    
    sample = dm.train_dataset[0]
    
    train_loader = dm.train_dataloader()
    batch = next(iter(train_loader))

    print("batch['x'].shape:", batch["x"].shape)
    print("batch['y'].shape:", batch["y"].shape)

   
    model = UNet(
        in_channels=len(INPUT_VARS),
        out_channels=len(TARGET_VARS),
    )

    
    with torch.no_grad():
        pred = model(batch["x"])

    print("pred.shape:", pred.shape)


if __name__ == "__main__":
    main()

Setting up DataModule...

Split sizes:
train: 7
val  : 2
test : 1

Checking one sample...
sample['x'].shape: torch.Size([20, 200, 255])
sample['y'].shape: torch.Size([1, 255])
sample['path']    : ../scripts/dataset/earthcare_02532E_GLM_6.h5

Checking one batch...
batch['x'].shape: torch.Size([4, 20, 200, 255])
batch['y'].shape: torch.Size([4, 1, 255])

Checking NaNs/Infs...

Building model...

Checking forward pass...
pred.shape: torch.Size([4, 1, 255])

Checking normalization stats...
